# Model Development
Walmart does not specify a required algorithm, modeling approach, technical constraints, or strict model requirements. Therefore, it is necesarry to select the most appropriate technologies based on the characteristics of the available data and the business context.

## Algorithm and Tool Selection

The available data volume is sufficient for classical and statistical time series models, but limited for deep learning approaches, which typically require significatly larger datasets to generalize well.

This approach allows models to be created almost instantaneously for each store. However, a potential limitation is that highly complex or non-linear patterns may not be fully captured, especially those that require large amounts of data to learn reliably.

### Models

Statistical models are typically easy to reproduce because they rely on deterministic estimation procedures with little or no stochastic componenets. As result, given the same data and configuration, they tend to produce consistent metrics and are generally fast to train. The following models are considered in next stages.

#### Naïve method
The [naïve forecasting method](https://openforecast.org/adam/simpleForecastingMethods.html#:~:text=3.3.1-,Na%C3%AFve,-Na%C3%AFve%20is%20one) assumes that the future value is equal to the last observed value.

$$
\hat{y}_{t} = y_{t-1}
$$

#### Seasonal Naïve method
Similar to Naïve, [Seasonal Naïve](https://openforecast.org/adam/simpleForecastingMethods.html#:~:text=3.3.6-,Seasonal%20Na%C3%AFve,-Finally%2C%20in%20the) relies only on one observation, but instead of taking the most recent value, it uses the value from the same period a season ago.
$$
\hat{y}_{t} = y_{t-m}
$$

where $m$ is the seasonal frequency.

#### ARIMA - Autoregressive Integrated Moving Average
[ARIMA](https://www.geeksforgeeks.org/machine-learning/arima-vs-sarima-model/#:~:text=SARIMA%20Model%20%2D%20FAQs-,What%20is%20ARIMA%20(Autoregressive%20Integrated%20Moving%20Average)%3F,-ARIMA%2C%20standing%20for) is a versatile model for analyzing and forecasting time series data.
It has three key components:
1. **Autoregression**: Captures the influence of a series' past values on its future values. It is denoted as 'p' which represents the number of lagged observations included in the model.
2. **Differencing**: Used to achieve stationarity by removing trends or other non-stationary patterns in the data. It is denoted by 'd', which indicates the number of times the series is differenced.
3. **Moving Average**: Captures the effect of past forecast errors (residuals) on the current prediction. It is denoted by 'q', which represents the number of lagged errors included in the model.

#### SARIMA - Seasonal Autoregressive Integrated Moving Average
[SARIMA](https://www.geeksforgeeks.org/machine-learning/arima-vs-sarima-model/#:~:text=refine%20future%20predictions.-,What%20is%20SARIMA(Seasonal%20Autoregressive%20Integrated%20Moving%20Average)%3F,-SARIMA%20(Seasonal%20ARIMA)) is Seasonal ARIMA incorportaing the seasonality dimension that is beneficial for data exhibiting recurring patterns at fixed intervals.
1. **Seasonal Autoregression**: Captures the influence of past seasonal values on future observations.
2. **Seasonal Differencing**: Removes repeating seasonal patterns from the data to help achieve stationarity.
3. **Seasonal Moving Average**: Captures the effect of past seasonal forecast errors on the current prediction.

#### ARIMAX and SARIMAX

ARIMAX and SARIMAX extend ARIMA and SARIMA by incorporating exogenous variables, allowing the model to use external information in the forecast, such as gold prices, oil prices, outdoor temperature, etc. 

By including external data, the model can respond more quickly and directly to changes in these factors, rather than relying solely on lagged values of the target time series to capture their effects.

### Tools
Since there are 45 stores and multiple candidate models, comparing results and tracking parameters and metrics manually can quickly become unmanageable

To ensure organized experimentation, reproducibility, and efficient model comparison, the following tools are used.

#### StatsForecasts
[StatsForcast](https://github.com/Nixtla/statsforecast) 2.0.3 is a library designed for large-scale time series forecasting and model benchmarking. It supports classical statistical models such as Naïve, Seasonal Naïve and the ARIMA family. One of its main advantages is native multi-series training, allowing multiple stores to be modeled simultaneously. It is primarily used to establish strong statistical baselines and to facilitate consistent model comparisons.

#### MLflow
[MLflow](https://mlflow.org) 3.9.0 is used for experimnet tracking and model management. It enables looging of key information for each training run, including hyperparameters, trained models, and evaluation metrics. Additionally, it provides a user interface that allows visual comparison of models withing the same experiment through charts and metrics dashboards.

## Model Training and Tunning

### Libraries and Imports

In [6]:
import time
import pickle
import mlflow
import pandas as pd
from datetime import datetime
from statsforecast import StatsForecast
from utilsforecast.losses import mae, rmse
from utilsforecast.evaluation import evaluate
from statsforecast.models import Naive, SeasonalNaive, AutoARIMA

### Model Input Preparation

In [4]:
walmart_ds = pd.read_parquet("../data/interim/walmart.parquet")
walmart_ds = walmart_ds.rename(columns={
    "Date":"ds", 
    "Weekly_Sales":"y", 
    "Store": "unique_id"})
walmart_ds = walmart_ds.sort_values(["unique_id", "ds"])

walmart_minimal = walmart_ds[["unique_id", "ds", "y"]].copy()
walmart_minimal.to_parquet("../data/processed/walmart_minimal.parquet")

walmart_no_new_flags = walmart_ds.drop(["Anomaly_Flag", "Structural_Change_Flag"], axis=1).copy()
walmart_no_new_flags.to_parquet("../data/processed/walmart_no_new_flags.parquet")

walmart_full = walmart_ds.copy()
walmart_full.to_parquet("../data/processed/walmart_full.parquet")

StatsForcast  requires a standarized input structure in order to train models across multiple time series. Specifically, the date column must be name `ds`, the target variable must be labeled `y`, and each time series must be identified using a `unique_id` column.

### Create a experiment

In [5]:
mlflow.set_tracking_uri("sqlite:///../mlflow/mlflow.db")
experiment_name = "walmart_models_experiment"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location="../mlflow/mlruns/"
    )
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_id=experiment_id)

<Experiment: artifact_location='file:///c:/Users/olver/Desktop/ml_walmart/notebooks/../mlflow/mlruns', creation_time=1771105746203, experiment_id='1', last_update_time=1771105746203, lifecycle_stage='active', name='walmart_models_experiment', tags={}>

An experiment named `walmart_models_experiment` will be created to track and store all model training runs.

### Iterative Training

`StatsForecast` will be used to train and automatically tune different forecasting models. For each run, the corresponding parameters and evaluation metrics will be logged using MLflow.

Regarding model management, `StatsForecast` does not provide built-in functionality to save trained models or retrieve the internal auto-tuning configurations. Therefore, additional libraries may be required for model persistence.

For evaluation, no manual data splitting is required because `StatsForecast` provides a built-in `cross_validation` method that handles rolling window evaluation internally.

In [7]:
shared_runtime = {
    "freq": "W",
    "n_jobs": -1
}

shared_cv = {
    "h": 12,
    "n_windows": 3,
}
configs = [
    {
        "input": walmart_minimal,
        "input_name": "walmart_minimal",
        "model": Naive(alias="Naive"),
        "name": "Naive",
        "model_params": None
    },
    {
        "input": walmart_minimal,
        "input_name": "walmart_minimal",
        "model": SeasonalNaive(alias="Seasonal_Naive", season_length=52),
        "name": "Seasonal_Naive",
        "model_params":{
            "season_length": 52
        }
    },
    {
        "input": walmart_minimal,
        "input_name": "walmart_minimal",
        "model": AutoARIMA(alias="ARIMA"),
        "name": "ARIMA",
        "model_params": None
    },
    {
        "input": walmart_minimal,
        "input_name": "walmart_minimal",
        "model": AutoARIMA(alias="SARIMA", season_length=52),
        "name": "SARIMA",
        "model_params": {
            "season_length": 52
        }
    },
    {
        "input": walmart_no_new_flags,
        "input_name": "walmart_no_new_flags",
        "model": AutoARIMA(alias="ARIMAX_No_New_Flags"),
        "name": "ARIMAX_No_New_Flags",
        "model_params": None
    },
    {
        "input": walmart_no_new_flags,
        "input_name": "walmart_no_new_flags",
        "model": AutoARIMA(alias="SARIMAX_No_New_Flags", season_length=52),
        "name": "SARIMAX_No_New_Flags",
        "model_params": {
            "season_length": 52
        }
    },
    {
        "input": walmart_full,
        "input_name": "walmart_full",
        "model": AutoARIMA(alias="ARIMAX_Full"),
        "name": "ARIMAX_Full",
        "model_params": None
    },
    {
        "input": walmart_full,
        "input_name": "walmart_full",
        "model": AutoARIMA(alias="SARIMAX_Full", season_length=52),
        "name": "SARIMAX_Full",
        "model_params": {
            "season_length": 52
        }
    }
]
def make_cross_validation(config, sf: StatsForecast, name: str):
    cv = sf.cross_validation(
        df=config["input"],
        **shared_cv
    )
    run_id = mlflow.active_run().info.run_id
    cv_path = f"../artifacts/cv_predictions/{name}_{run_id}.csv"
    cv.to_csv(cv_path, index = False)
    mlflow.log_artifact(cv_path)
    return cv

def evaluate_cross_validation(cv: pd.DataFrame, name: str):
    eval_df = evaluate(cv, metrics=[mae, rmse])
    run_id = mlflow.active_run().info.run_id
    eval_path = f"../artifacts/metrics/{name}_{run_id}.csv"
    eval_df.to_csv(eval_path, index=False)
    mlflow.log_artifact(eval_path)
    return eval_df

def log_evaluation_metrics(config, eval_df: pd.DataFrame, name: str):
    mae_df = eval_df[eval_df["metric"] == "mae"]
    rmse_df = eval_df[eval_df["metric"] == "rmse"]
    mae_val = mae_df[name].mean()
    rmse_val = rmse_df[name].mean()
    mlflow.log_metric("mae", mae_val)
    mlflow.log_metric("rmse", rmse_val)
    for _, row in mae_df.iterrows():
        store_id = row["unique_id"]
        mlflow.log_metric(f"mae_store_{store_id}", row[name])
    for _, row in rmse_df.iterrows():
        store_id = row["unique_id"]
        mlflow.log_metric(f"rmse_store_{store_id}", row[name])
    print(config["name"], f"mae: {mae_val}", f"rmse: {rmse_val}")

def fit_and_save_model(config, sf:StatsForecast, name: str):
    sf.fit(config["input"])
    run_id = mlflow.active_run().info.run_id
    model_path = f"../artifacts/models/{name}_{run_id}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(sf, f)
    mlflow.log_artifact(model_path)


for config in configs:
    with mlflow.start_run(run_name=config["name"]):
        start = time.time()
        mlflow.log_params({
            "model": config["name"],
            **(config["model_params"] or {}),
            "dataset": config["input_name"],
            **shared_runtime,
            **shared_cv
        })
        sf = StatsForecast(
            models=[config["model"]],
            **shared_runtime
        )
        cv = make_cross_validation(config, sf, config["name"])
        eval_df = evaluate_cross_validation(cv, config["name"])
        log_evaluation_metrics(config, eval_df,config["name"])
        fit_and_save_model(config, sf, config["name"])
        elapsed = time.time() - start
        mlflow.log_metric("fit_time_sec", elapsed)

Naive mae: 60763.000487654324 rmse: 73194.87680775565
Seasonal_Naive mae: 53936.254382716055 rmse: 64120.49563801813
ARIMA mae: 55105.18786685701 rmse: 65726.54912217768
SARIMA mae: 39870.87053591712 rmse: 50355.78279793694
ARIMAX_No_New_Flags mae: 62238.23200465143 rmse: 72213.8920866113
SARIMAX_No_New_Flags mae: 49093.39915891323 rmse: 59709.70404558067
ARIMAX_Full mae: 62142.67661633561 rmse: 71964.0936574705
SARIMAX_Full mae: 44882.72932836544 rmse: 54490.037804017426


`configs` contains the model configurations for each training run. In every loop iteration, the process logs parameters, trains the model, performs cross-validation, evaluates the cross-validation results, and logs the corresponding metrics. In parallel It shows Mean Absolute Error and Root Mean Squared Error.